# Week 4: Exploratory Data Analysis

**Foundations of Data Science**

## In this lesson

1. What EDA is for: generating hypotheses, not confirming them
2. One variable at a time
3. Two variables: numeric with numeric, categorical with outcome, numeric by group
4. Three or more: where the interesting things hide
5. Confounding, the trap EDA exists to catch
6. The findings log: writing down what you saw and what you'd test

## The one idea

**EDA is structured curiosity with a notebook open.** You look at every variable, then every pair, then the combinations that the pairs suggest, and you write down what surprised you. Nothing you find in EDA is a result; it's a list of things worth testing. The analyst who skips EDA fits a model to data they don't understand; the analyst who never leaves EDA finds a pattern in every noise.

The clinic question widens today. Age and weight predict diabetes; so what else does, and does anything change the age story? We'll look at sex, race/ethnicity, income, waist, and blood pressure.

## Setup

<!-- nb-links -->
[Download this notebook](week_04_eda.ipynb?download=1){download="week_04_eda.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/week_04/week_04_eda.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/week_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("week_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this week's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Load, label, and look once

Codes to labels first. EDA is about reading plots, and a plot labeled `race_eth = 3` can't be read. The CDC codebook for `RIDRETH3`: 1 Mexican American, 2 Other Hispanic, 3 Non-Hispanic White, 4 Non-Hispanic Black, 6 Non-Hispanic Asian, 7 Other or multiracial. Income ratio is family income divided by the federal poverty line, capped at 5.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()

adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({
    1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"
})
adults["age_band"] = pd.cut(adults["age"], bins=[20, 40, 60, 81], labels=["20-39", "40-59", "60+"], right=False).astype(str)
adults["bmi_cat"] = pd.cut(adults["bmi"], bins=[0, 25, 30, 100], labels=["normal", "overweight", "obese"], right=False).astype(str)
adults["income_band"] = pd.cut(adults["income_ratio"], bins=[0, 1, 2, 5.01], labels=["below poverty", "1-2x", "2x+"], right=False).astype(str)
adults["diabetes_label"] = adults["diabetes"].map({0: "no diabetes", 1: "diabetes"})

numeric = ["age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"]
categorical = ["sex", "race_eth", "age_band", "bmi_cat", "income_band"]

from eda_toolkit import dataframe_profiler
dataframe_profiler(adults[numeric + categorical + ["diabetes"]], return_df=True)

## 2. One variable at a time

Every numeric column gets a histogram; every categorical column gets a count. You're looking for: shape (skew, bimodality), impossible values, categories too small to say anything about, and anything that doesn't match what you expected from the codebook.

In [ ]:
from eda_toolkit import plot_distributions

plot_distributions(
    adults, vars_of_interest=numeric,
    n_rows=2, n_cols=3, subplot_figsize=(15, 8),
    plot_mean=True, plot_median=True, mean_color="red", median_color="black",
)

In [ ]:
# Categorical counts. The 'Other/multi' group and some income cells are small; keep that in mind for every split below.
for col in categorical:
    print(adults[col].value_counts(dropna=False).to_string(), "\n")

Notes from the univariate pass (write yours as you go):
- BMI, waist, and SBP are right-skewed, as established last week. `income_ratio` piles up at 5 because it's capped there; that's a design artifact, not a real mode.
- `Other/multi` has 273 people, `below poverty` 878. Any split of those by age band and sex will have thin cells.

## 3. Two variables

Three kinds of pair, three kinds of plot.

**Numeric with numeric: correlation.** Pearson's r measures how close the points are to a straight line (-1 to 1). Spearman's version does the same on ranks, so it's robust to skew and outliers and catches monotone curves that aren't lines. For skewed data like ours, Spearman is the safer default. A correlation of 0.9 between BMI and waist says they carry nearly the same information; a model doesn't need both.

In [ ]:
from eda_toolkit import flex_corr_matrix

flex_corr_matrix(
    adults, cols=numeric, corr_method="spearman",
    figsize=(7, 6), title="Spearman correlations, adults",
    show_significance=True, show_plot=True,
)

In [ ]:
from eda_toolkit import scatter_fit_plot

# Two pairs worth a closer look: the near-duplicate (BMI, waist) and the age-pressure relationship, colored by outcome.
scatter_fit_plot(
    adults.dropna(subset=["bmi", "waist_cm", "sbp"]),
    x_vars=["bmi", "age"], y_vars=["waist_cm", "sbp"],
    exclude_combinations=[("bmi", "sbp"), ("age", "waist_cm")],
    hue="diabetes_label", add_best_fit_line=True, show_correlation=True,
    n_rows=1, n_cols=2, subplot_figsize=(13, 5), **{"alpha": 0.4, "s": 12},
)

**Categorical with outcome: crosstab bars.** For each categorical variable, what share of each level has the outcome? This is the plot version of the week 1 prevalence table, for every variable at once.

In [ ]:
from eda_toolkit import outcome_crosstab_plot

outcome_crosstab_plot(
    adults, list_name=["sex", "age_band", "bmi_cat", "race_eth", "income_band"],
    outcome="diabetes", normalize=True, label_0="no diabetes", label_1="diabetes",
    n_rows=2, n_cols=3, figsize=(16, 9),
)

**Numeric by group: box plots.** Same variable, split by a category. Compare the middles *and* the spreads; two groups with the same median can have very different tails.

In [ ]:
from eda_toolkit import box_violin_plot

box_violin_plot(
    adults.dropna(subset=["bmi", "waist_cm", "sbp"]),
    metrics_list=["bmi", "waist_cm", "sbp"], metrics_comp=["diabetes_label", "age_band"],
    plot_type="boxplot", n_rows=2, n_cols=3, subplot_figsize=(15, 8), show_plot="subplots",
)

## 4. Three or more variables

Pairs tell you what's associated with the outcome. Triples tell you whether that association survives once you hold something else fixed. The most useful triple is almost always `outcome x variable x the strongest predictor you already know`, and here the strongest predictor is age.

Before you build the table, check the cell sizes. A prevalence computed on 15 people is noise with a decimal point.

In [ ]:
from eda_toolkit import summarize_all_combinations

# Count and share of adults in every combination of these three. Look for any cell under ~50 before trusting a rate from it.
combos, keys = summarize_all_combinations(adults, variables=["race_eth", "age_band", "sex"], min_length=3)
three_way = combos[keys[0]]
print("smallest cells:")
three_way.sort_values("Count").head(6)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prevalence by race/ethnicity within age band, as a heatmap. Compare each row to the crude (all-ages) column.
pivot = adults.pivot_table(index="race_eth", columns="age_band", values="diabetes", aggfunc="mean")
pivot["all ages"] = adults.groupby("race_eth")["diabetes"].mean()
mean_age = adults.groupby("race_eth")["age"].mean().round(1)
pivot.index = [f"{r}  (mean age {mean_age[r]})" for r in pivot.index]

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=0.4, ax=ax, cbar_kws={"label": "share with diabetes"})
ax.set_title("Diabetes prevalence by race/ethnicity: crude vs within age band")
ax.set_ylabel("")
plt.show()

## 5. Confounding: the trap EDA exists to catch

Read the heatmap row by row against the last column. In the crude numbers, White adults sit in the middle of the pack (16%). Within every age band they are at or near the bottom; at 60+ it's 25% against 38% for Mexican American adults. The crude number hides this because White respondents are the oldest group by five to seven years, and age is the strongest driver of diabetes. **Age is a confounder**: it's associated with both the exposure (group) and the outcome, so the crude comparison mixes the two.

The general rule: before you believe any association between X and the outcome, split by the strongest known predictor and see if it holds. If it strengthens, weakens, or reverses, you've learned something that a correlation matrix could never tell you. The same thing happens with income below.

In [ ]:
# Same check for income: crude vs within age band
inc = adults[adults["income_band"] != "nan"]
crude = inc.groupby("income_band")["diabetes"].mean().rename("all ages")
by_age = inc.pivot_table(index="income_band", columns="age_band", values="diabetes", aggfunc="mean")
pd.concat([by_age, crude], axis=1).loc[["below poverty", "1-2x", "2x+"]].round(3)

Crude: no clear gradient, and the middle group is highest. Within each age band: the 2x+ group has the lowest prevalence, consistently, and the two poorer groups match each other. The crude table hides it because the below-poverty group is the youngest (mean age 49 against 52 for the other two), which drags its all-ages number down. Same mechanism as race, different direction.

Neither of these is a causal claim. Adjusting for age in a table is the first step; a regression in week 6 adjusts for several things at once; and even then, "associated with" is as far as survey data go.

## 6. The findings log

Nothing above is a result. It's a list of hypotheses, each with the week we'll test it properly:

| What EDA showed | Hypothesis | Where it gets tested |
|---|---|---|
| BMI and waist correlate at 0.91 | one of them is redundant in a model | week 6 (collinearity), week 7 (feature selection) |
| SBP rises with age (r = 0.49) and with BMI (0.14) | SBP adds little beyond age for predicting diabetes | week 7 |
| Men have higher prevalence at 40+, women slightly higher at 20-39 | sex matters, and its effect may depend on age (an interaction) | week 6 (interaction terms), week 7 |
| Race/ethnicity gaps appear only after age adjustment | group effects are real but smaller than age; needs adjusted estimates | week 6, week 13 (fairness) |
| Income gradient appears only within age bands | same | week 6 |
| `Other/multi` and `below poverty x 60+ x male` cells are small | estimates for those cells need intervals, or pooling | every week |

Keep a table like this for your project. The one-page problem statement due today should end with the first two or three rows of yours.

## Looking ahead
- **Project**: with the problem statement done, keep the findings log growing; it feeds next week's chart.
- **Reading**: VanderPlas chapter 4 (matplotlib) in full; it's the basis for next week. Bruce & Bruce chapter 1, the EDA sections, for a second opinion on today.